In [1]:
import os
import torch
import numpy as np
import xarray as xr
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader

from src.dataset import LazyWeatherDataset
from src.preprocessing import flatten_target_dataset, standardize_with_stats, compute_overall_from_daily_stats
from src.models import get_model

In [2]:
# ---------------------------
# IG routine
# ---------------------------
def integrated_gradients(model, x, baseline, scalar_fn, steps=32):
    """
    x, baseline: [B, C, H, W, T]
    returns IG tensor same shape
    """
    B = x.shape[0]
    device = x.device

    alphas = torch.linspace(0, 1, steps, device=device).view(steps, 1, 1, 1, 1, 1)

    # Create interpolated path: [steps, B, C, H, W, T]
    path = baseline.unsqueeze(0) + alphas * (x.unsqueeze(0) - baseline.unsqueeze(0))

    # Collapse steps into batch: [steps*B, C, H, W, T]
    path = path.view(steps * B, *x.shape[1:])
    path.requires_grad_(True)

    outputs = scalar_fn(model(path))  # [steps*B]
    grads = torch.autograd.grad(outputs.sum(), path)[0]

    # Restore shape: [steps, B, C, H, W, T]
    grads = grads.view(steps, B, *x.shape[1:])

    avg_grads = grads.mean(dim=0)  # [B, C, H, W, T]
    ig = (x - baseline) * avg_grads

    return ig.detach()


# ---------------------------
# SHASH scalar targets
# ---------------------------

def shash_scalars(pred_params, target_idx):
    """
    pred_params: [B, K*4]
    returns dict of scalar tensors [B]
    """
    K = pred_params.shape[1] // 4
    params = pred_params.view(-1, K, 4)

    mu = params[:, :, 0]
    sigma = torch.exp(params[:, :, 1])

    mu_t = mu[:, target_idx]
    sig_t = sigma[:, target_idx]

    prob_hi = 1 - 0.5 * (1 + torch.erf((2 - mu_t) / (sig_t * np.sqrt(2))))
    prob_lo = 0.5 * (1 + torch.erf((-2 - mu_t) / (sig_t * np.sqrt(2))))

    return {
        "mean": mu_t,
        "sigma": sig_t,
        "p_gt_2": prob_hi,
        "p_lt_-2": prob_lo,
    }


# ---------------------------
# Baseline builder
# ---------------------------

def build_climatology(loader, device):
    total = None
    count = 0

    for xb, _ in loader:
        xb = xb.to(device)
        if total is None:
            total = xb.sum(dim=0, keepdim=True)
        else:
            total += xb.sum(dim=0, keepdim=True)
        count += xb.shape[0]

    return total / count  # [1, C, H, W, T]


# ---------------------------
# Main driver
# ---------------------------

def run_ig_for_model(model_name, level, inputs, targets, stats,
                     batch_size=32, steps=32, latest=False, n_splits=5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    days = inputs.day.values
    kf = KFold(n_splits=n_splits, shuffle=False)

    overall_stats = compute_overall_from_daily_stats(stats)
    input_dims = 5

    for fold, (train_idx, val_idx) in enumerate(kf.split(days)):

        print(f"\n=== Fold {fold} ===")

        train_days = days[train_idx]
        val_days = days[val_idx]

        X_train = inputs.sel(day=train_days)
        X_val = inputs.sel(day=val_days)

        fold_stats = compute_overall_from_daily_stats(stats.sel(day=train_days))

        conversion_stats = xr.Dataset({
            v: ((fold_stats[v] - overall_stats[v]) / overall_stats[v.replace('_mean', '_std')])
            if v.endswith('_mean')
            else (fold_stats[v] / overall_stats[v])
            for v in fold_stats.data_vars
        })

        X_train_std = standardize_with_stats(X_train, conversion_stats)
        X_val_std = standardize_with_stats(X_val, conversion_stats)

        train_ds = LazyWeatherDataset(
            X_train_std,
            y=flatten_target_dataset(targets.sel(time=train_days)),
            input_dimensions=5
        )

        val_ds = LazyWeatherDataset(
            X_val_std,
            y=flatten_target_dataset(targets.sel(time=val_days)),
            input_dimensions=5
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size)
        val_loader = DataLoader(val_ds, batch_size=1)

        # ---- Build baseline ----
        print("Building baseline climatology...")
        baseline = build_climatology(train_loader, device)

        # ---- Load model ----
        model = get_model(model_name.split('/')[0], next(iter(train_loader))[0].shape[1:], 36, targets=None).to(device)

        model_path = f"models/{model_name}/fold={fold}/" + ("latest.pt" if latest else "best.pt")
        model.load_state_dict(torch.load(model_path, map_location="cpu")["model_state_dict"])
        model.eval()

        # ---- Accumulators ----
        C, H, W, T = baseline.shape[1:]
        K = 9
        scalars = ["mean", "sigma", "p_gt_2", "p_lt_-2"]

        chan_acc = {s: torch.zeros(K, C) for s in scalars}
        spat_acc = {s: torch.zeros(K, H, W) for s in scalars}
        temp_acc = {s: torch.zeros(K, T) for s in scalars}

        n_days = 0

        # ---- IG loop ----
        for xb, _ in val_loader:
            xb = xb.to(device)
            n_days += 1

            for target_idx in range(K):

                for scalar_name in scalars:
                    def scalar_fn(out, sn=scalar_name, ti=target_idx):
                        return shash_scalars(out, ti)[sn]
                        
                    ig = integrated_gradients(model, xb, baseline, scalar_fn, steps=steps)[0].detach().cpu()

                    chan_acc[scalar_name][target_idx] += ig.abs().sum(dim=(1, 2, 3))
                    spat_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0, 3))
                    temp_acc[scalar_name][target_idx] += ig.abs().sum(dim=(0, 1, 2))

        # ---- Normalize ----
        for s in scalars:
            chan_acc[s] /= n_days
            spat_acc[s] /= n_days
            temp_acc[s] /= n_days

        # ---- Save ----
        out_dir = f"results/ig/{model_name}/fold_{fold}"
        os.makedirs(out_dir, exist_ok=True)

        torch.save({
            "channel": chan_acc,
            "spatial": spat_acc,
            "temporal": temp_acc
        }, os.path.join(out_dir, "ig_results.pt"))

        print(f"Saved fold {fold}")

In [3]:
inputs = xr.open_zarr("/glade/work/milesep/convective_outlook_ml/train_inputs_slgt_small_glade.zarr")
targets = xr.open_dataset("data/processed_data/train_targets_slgt_new.nc")
stats = xr.open_dataset("data/processed_data/daily_input_stats_slgt_small_glade.nc")

In [ ]:
inputs.nbytes

In [29]:
run_ig_for_model(
    model_name='cnn3d_gelu_0_5/level=slgt_small_glade_new/opt=Adam_lr=0.001_batch=8_crit=ShashNLL',
    level='slgt_small_glade_new',
    inputs=inputs,
    targets=targets,
    stats=stats,
    steps=32,
)


=== Fold 0 ===
Building baseline climatology...
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265